In [8]:
import pandas as pd
import numpy as np
import joblib
from itertools import combinations

print("Imports done!")

Imports done!


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Save your account (only need to do this once)
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="your_token_here",
    overwrite=True
)
print("Account saved!")

Account saved!


In [10]:
df = pd.read_csv('uniprot_proteins.tsv', sep='\t')
df_clean = df[df['Interacts with'].notna()].reset_index(drop=True)
df_clean['n_interactions'] = df_clean['Interacts with'].str.split(';').apply(len)
print(df.shape)
print(df_clean.shape)

(500, 6)
(377, 7)


In [11]:
print(df['Interacts with'].isna().sum())
print(df.columns.tolist())

123
['Entry', 'Protein names', 'Length', 'Helix', 'Beta strand', 'Interacts with']


In [12]:
df_clean = df[df['Interacts with'].notna()].reset_index(drop=True)
print(df_clean.shape)
print(df_clean[['Entry', 'Length', 'Interacts with']].head(3))

(377, 6)
    Entry  Length                                     Interacts with
0  A0JP26     581  Q08AG9; O95995; Q9BYQ6; Q9BYR2; P26371; Q9BYQ4...
1  A0PK11     232  Q8N6S5; O00501; Q9UHP7-3; Q8TBE3; P26715; Q8N3...
2  A1A4S6     786                                     Q9UNA1; Q6P5Z2


In [13]:
df_clean['n_interactions'] = df_clean['Interacts with'].str.split(';').apply(len)
print(df_clean[['Entry', 'Length', 'n_interactions']].head(5))

    Entry  Length  n_interactions
0  A0JP26     581               8
1  A0PK11     232              12
2  A1A4S6     786               2
3  A1L190      88              10
4  A1L3X0     281              11


In [14]:

from itertools import combinations

pairs = []
proteins = df_clean['Entry'].tolist()

# FAST - dictionary lookup is O(1)
protein_dict = df_clean.set_index('Entry').to_dict('index')  # built ONCE

for p1, p2 in combinations(proteins, 2):
    row1 = protein_dict[p1]                                   # O(1) lookup
    row2 = protein_dict[p2]                                   # O(1) lookup
    
    length_diff = abs(row1['Length'] - row2['Length'])
    n_int_sum = row1['n_interactions'] + row2['n_interactions']
    
    partners1 = set(row1['Interacts with'].split('; '))
    partners2 = set(row2['Interacts with'].split('; '))
    shared = int(len(partners1 & partners2) > 0)
    
    pairs.append([p1, p2, row1['Length'], row2['Length'],
                  length_diff, n_int_sum, shared])

pairs_df = pd.DataFrame(pairs, columns=[
    'Protein1', 'Protein2', 'Length1', 'Length2',
    'length_diff', 'n_int_sum', 'interacts'
])

print(pairs_df.shape)
print(pairs_df['interacts'].value_counts())
print(pairs_df.head(3))

(70876, 7)
interacts
0    68283
1     2593
Name: count, dtype: int64
  Protein1 Protein2  Length1  Length2  length_diff  n_int_sum  interacts
0   A0JP26   A0PK11      581      232          349         20          0
1   A0JP26   A1A4S6      581      786          205         10          0
2   A0JP26   A1L190      581       88          493         18          0


In [15]:
df = pd.read_csv('uniprot_proteins.tsv', sep='\t')
df_clean = df[df['Interacts with'].notna()].reset_index(drop=True)
df_clean['n_interactions'] = df_clean['Interacts with'].str.split(';').apply(len)

protein_dict = df_clean.set_index('Entry').to_dict('index')
proteins = df_clean['Entry'].tolist()
pairs = []

for p1, p2 in combinations(proteins, 2):
    row1 = protein_dict[p1]
    row2 = protein_dict[p2]
    length_diff = abs(row1['Length'] - row2['Length'])
    n_int_sum = row1['n_interactions'] + row2['n_interactions']
    partners1 = set(row1['Interacts with'].split('; '))
    partners2 = set(row2['Interacts with'].split('; '))
    shared = int(len(partners1 & partners2) > 0)
    pairs.append([p1, p2, row1['Length'], row2['Length'],
                  length_diff, n_int_sum, shared])

pairs_df = pd.DataFrame(pairs, columns=[
    'Protein1', 'Protein2', 'Length1', 'Length2',
    'length_diff', 'n_int_sum', 'interacts'
])

# Save immediately this time
pairs_df.to_csv('results/pairs_df.csv', index=False)
print(pairs_df.shape)

(70876, 7)


In [16]:

print(pairs_df['interacts'].value_counts())

interacts
0    68283
1     2593
Name: count, dtype: int64


In [17]:
rf = joblib.load('results/rf_model.pkl')
scaler = joblib.load('results/scaler.pkl')
print("Ready!")

Ready!


In [18]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit.library import GroverOperator
from qiskit.quantum_info import Statevector

print("Qiskit ready!")

Qiskit ready!


In [19]:
X = pairs_df[['Length1', 'Length2', 'length_diff', 'n_int_sum']]

# Get interaction probability scores from RF
pairs_df['rf_score'] = rf.predict_proba(X)[:, 1]

# Take top 1000 pairs by score
top1000 = pairs_df.nlargest(1000, 'rf_score').reset_index(drop=True)

# Target = index of highest scoring pair
target_index = 0  # after nlargest, best pair is always index 0

print(f"Target pair: {top1000.loc[0, 'Protein1']} <-> {top1000.loc[0, 'Protein2']}")
print(f"RF score: {top1000.loc[0, 'rf_score']:.4f}")
print(f"Total pairs to search: {len(top1000)}")

Target pair: A0JP26 <-> A8MW99
RF score: 1.0000
Total pairs to search: 1000


In [20]:
from qiskit import QuantumCircuit

n_qubits = 10
target = 0  # index of best pair in binary

def build_oracle(n_qubits, target_index):
    oracle = QuantumCircuit(n_qubits)
    
    # Convert target index to binary string
    target_bin = format(target_index, f'0{n_qubits}b')
    print(f"Target in binary: {target_bin}")
    
    # Flip qubits where target bit is 0
    for i, bit in enumerate(target_bin):
        if bit == '0':
            oracle.x(i)
    
    # Multi-controlled Z gate
    oracle.h(n_qubits - 1)
    oracle.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    oracle.h(n_qubits - 1)
    
    # Flip back
    for i, bit in enumerate(target_bin):
        if bit == '0':
            oracle.x(i)
    
    return oracle

oracle = build_oracle(n_qubits, target)
print(oracle)

Target in binary: 0000000000
     ┌───┐          ┌───┐     
q_0: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_1: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_2: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_3: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_4: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_5: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_6: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_7: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_8: ┤ X ├───────■──┤ X ├─────
     ├───┤┌───┐┌─┴─┐├───┤┌───┐
q_9: ┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├
     └───┘└───┘└───┘└───┘└───┘


In [21]:
def build_diffusion(n_qubits):
    diffusion = QuantumCircuit(n_qubits)
    
    # Apply H to all qubits
    diffusion.h(range(n_qubits))
    
    # Apply X to all qubits
    diffusion.x(range(n_qubits))
    
    # Multi-controlled Z (same H-MCX-H trick)
    diffusion.h(n_qubits - 1)
    diffusion.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    diffusion.h(n_qubits - 1)
    
    # Apply X to all qubits
    diffusion.x(range(n_qubits))
    
    # Apply H to all qubits
    diffusion.h(range(n_qubits))
    
    return diffusion

diffusion = build_diffusion(n_qubits)
print(diffusion)

     ┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_2: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_3: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_4: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_5: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_6: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_7: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_8: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐
q_9: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘


In [22]:
from qiskit_aer import AerSimulator
import math

# Number of Grover iterations
n_iterations = int(math.pi / 4 * math.sqrt(1000))
print(f"Grover iterations: {n_iterations}")

# Build full circuit
grover = QuantumCircuit(n_qubits, n_qubits)

# Step 1: Superposition
grover.h(range(n_qubits))

# Step 2: Repeat oracle + diffusion
for _ in range(n_iterations):
    grover = grover.compose(oracle)
    grover = grover.compose(diffusion)

# Step 3: Measure
grover.measure(range(n_qubits), range(n_qubits))

# Run on AerSimulator
simulator = AerSimulator()
job = simulator.run(grover, shots=1024)
result = job.result()
counts = result.get_counts()

# Get most frequent measurement
top_result = max(counts, key=counts.get)
found_index = int(top_result, 2)

print(f"Most measured state : {top_result}")
print(f"Corresponds to index: {found_index}")
print(f"Target index was    : {target_index}")
print(f"Match: {found_index == target_index}")

Grover iterations: 24
Most measured state : 0000000000
Corresponds to index: 0
Target index was    : 0
Match: True


In [23]:
import numpy as np

n_pairs = len(top1000)

classical_steps = n_pairs
grover_steps = int(math.pi / 4 * math.sqrt(n_pairs))
speedup = classical_steps / grover_steps

print("=" * 45)
print("  CLASSICAL vs GROVER'S SEARCH COMPARISON")
print("=" * 45)
print(f"  Dataset size      : {n_pairs} pairs")
print(f"  Classical O(N)    : {classical_steps} steps")
print(f"  Grover's O(√N)    : {grover_steps} steps")
print(f"  Speedup           : {speedup:.1f}x")
print()

# Theoretical for full dataset
n_full = len(pairs_df)
grover_full = int(math.pi / 4 * math.sqrt(n_full))
speedup_full = n_full / grover_full

print("  --- Theoretical (full 70,876 pairs) ---")
print(f"  Classical O(N)    : {n_full} steps")
print(f"  Grover's O(√N)    : {grover_full} steps")
print(f"  Speedup           : {speedup_full:.1f}x")
print("=" * 45)

  CLASSICAL vs GROVER'S SEARCH COMPARISON
  Dataset size      : 1000 pairs
  Classical O(N)    : 1000 steps
  Grover's O(√N)    : 24 steps
  Speedup           : 41.7x

  --- Theoretical (full 70,876 pairs) ---
  Classical O(N)    : 70876 steps
  Grover's O(√N)    : 209 steps
  Speedup           : 339.1x


In [24]:
# Save results
import json

results = {
    "classical_steps_demo": 1000,
    "grover_steps_demo": 24,
    "speedup_demo": 41.7,
    "classical_steps_full": 70876,
    "grover_steps_full": 209,
    "speedup_full": 339.1,
    "target_pair": ["A0JP26", "A8MW99"],
    "rf_score": 1.0,
    "rf_f1": 0.22,
    "rf_recall": 0.36
}

with open('results/final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved!")

Saved!


In [25]:
print("=== ORACLE CIRCUIT ===")
print(oracle.draw('text'))

print("\n=== DIFFUSION CIRCUIT ===")
print(diffusion.draw('text'))

=== ORACLE CIRCUIT ===
     ┌───┐          ┌───┐     
q_0: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_1: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_2: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_3: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_4: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_5: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_6: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_7: ┤ X ├───────■──┤ X ├─────
     ├───┤       │  ├───┤     
q_8: ┤ X ├───────■──┤ X ├─────
     ├───┤┌───┐┌─┴─┐├───┤┌───┐
q_9: ┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├
     └───┘└───┘└───┘└───┘└───┘

=== DIFFUSION CIRCUIT ===
     ┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_2: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_3: ┤ H ├┤ 

In [26]:
with open('oracle_circuit.txt', 'w', encoding='utf-8') as f:
    f.write(oracle.draw('text').single_string())
    
with open('diffusion_circuit.txt', 'w', encoding='utf-8') as f:
    f.write(diffusion.draw('text').single_string())

print("Saved!")

Saved!


In [27]:
service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=10, operational=True)
print(f"Using backend: {backend.name}")
print(f"Qubits: {backend.num_qubits}")

qiskit_runtime_service.__init__:WARNING:2026-05-25 10:31:43,375: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Protein protein interaction. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-25 10:31:44,667: Loading instance: Protein protein interaction, plan: open
qiskit_runtime_service.backends:WARNING:2026-05-25 10:31:48,601: Using instance: Protein protein interaction, plan: open


Using backend: ibm_kingston
Qubits: 156


In [28]:
from qiskit.compiler import transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Transpile circuit for real hardware
transpiled = transpile(grover, backend=backend, optimization_level=3)
print(f"Original gates : {sum(grover.count_ops().values())}")
print(f"Transpiled gates: {sum(transpiled.count_ops().values())}")
print(f"Circuit depth  : {transpiled.depth()}")

Original gates : 1604
Transpiled gates: 172024
Circuit depth  : 93575


In [29]:
n_qubits_small = 5
target_index = 0
n_iterations_small = int(math.pi/4 * math.sqrt(32))
print(f"Iterations: {n_iterations_small}")

# Build circuits
oracle_small = build_oracle(n_qubits_small, target_index)
diffusion_small = build_diffusion(n_qubits_small)

# Build full Grover circuit
grover_small = QuantumCircuit(n_qubits_small, n_qubits_small)
grover_small.h(range(n_qubits_small))

for _ in range(n_iterations_small):
    grover_small = grover_small.compose(oracle_small)
    grover_small = grover_small.compose(diffusion_small)

grover_small.measure(range(n_qubits_small), range(n_qubits_small))
print("Circuit built!")

Iterations: 4
Target in binary: 00000
Circuit built!


In [30]:
transpiled_small = transpile(grover_small, backend=backend, optimization_level=3)
print(f"Original gates  : {sum(grover_small.count_ops().values())}")
print(f"Transpiled gates: {sum(transpiled_small.count_ops().values())}")
print(f"Circuit depth   : {transpiled_small.depth()}")

Original gates  : 154
Transpiled gates: 2302
Circuit depth   : 1423


In [32]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(backend)
job = sampler.run([transpiled_small], shots=1024)
print(f"Job ID: {job.job_id()}")
print("Job submitted! Waiting in queue...")
result = job.result()
print("Done!")

Job ID: d89vjj8p0eas73dp6fh0
Job submitted! Waiting in queue...
Done!


In [ ]:
print(f"Job status: {job.status()}")

In [33]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
job = service.job("d89vjj8p0eas73dp6fh0")
print(job.status())
result = job.result()

qiskit_runtime_service.__init__:WARNING:2026-05-25 15:23:52,948: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Protein protein interaction. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


DONE


In [38]:
counts = result[0].data.c.get_counts()
top_result = max(counts, key=counts.get)
found_index = int(top_result, 2)

print(f"Most measured state : {top_result}")
print(f"Corresponds to index: {found_index}")
print(f"Target was          : 0")
print(f"Match               : {found_index == 0}")
print(f"\nTop 5 measurements:")
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
for state, count in sorted_counts[:5]:
    print(f"  {state}: {count} shots")

Most measured state : 00000
Corresponds to index: 0
Target was          : 0
Match               : True

Top 5 measurements:
  00000: 176 shots
  00001: 46 shots
  01000: 42 shots
  00011: 37 shots
  01110: 35 shots


In [37]:
print(result[0].data)
print(dir(result[0].data))

DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=5>))
['_FIELDS', '_FIELD_TYPES', '_RESTRICTED_NAMES', '_SHAPE', '__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_data', '_is_protocol', '_is_runtime_protocol', '_shape', 'c', 'items', 'keys', 'ndim', 'shape', 'size', 'values']


In [39]:
import json
with open('results/ibm_quantum_results.json', 'w') as f:
    json.dump(counts, f)
print("Saved!")

Saved!
